# Structured execution

A statevector simulator pays one pass over all $2^n$ amplitudes per gate.
Many circuits are built from pieces whose *meaning* is much cheaper than
their gates: an adder or a streaming step only moves amplitudes between
basis states, a small block acts on a handful of qubits, and phase
estimation applies the same controlled unitary over and over.

`Block.statevector` and `QarpEngine` read the block tree before simulating
and run each piece as the cheapest exact kernel — a permutation as one
gather, a small block as one dense matrix, a controlled-power ladder as one
kernel — and everything else as its (fused) gates.  Results are exact,
global phase included; a circuit with no such structure runs exactly as
before.

## A permutation, one gather

An increment $x \to x + 1 \bmod 2^k$ written as a ladder of multi-controlled
flips.  Repeated on two interleaved registers of a 16-qubit state it is
hundreds of gates, but as a whole it only permutes basis states.

In [ ]:
import time

import numpy as np

from qarp.blocks import CompositeBlock, SimpleBlock


class Increment(SimpleBlock):
    def build_vanilla(self):
        for t in reversed(range(1, self.n_qubits)):
            self.mcx(*range(t), t)
        self.x(0)


n = 16
step = CompositeBlock(
    [Increment(8, target_qubits=list(range(0, n, 2))), Increment(8, target_qubits=list(range(1, n, 2)))],
    n_qubits=n,
)
block = (step ** 8).build()

rng = np.random.default_rng(0)
psi = rng.standard_normal(1 << n) + 1j * rng.standard_normal(1 << n)
psi /= np.linalg.norm(psi)

t = time.perf_counter()
gates = block.statevector(psi, structured=False)
t_gates = time.perf_counter() - t
block.statevector(psi, structured=True)  # plans once; the plan is cached on the block
t = time.perf_counter()
structured = block.statevector(psi, structured=True)
t_structured = time.perf_counter() - t

print(f"{len(block.flatten())} gates: {t_gates * 1e3:.1f} ms as gates, {t_structured * 1e3:.2f} ms structured")
print("max |difference|:", np.max(np.abs(gates - structured)))

## Declaring the action

A block that knows it is a permutation can say so with `classical_action`:
it maps local basis indices to their images (int64, LSB first).  The
declaration is a promise — $U|x\rangle = |f(x)\rangle$ exactly, phase
included — and `ModularMultiplicationBlock` makes it for $x \to m x \bmod N$.

In [ ]:
from qarp.blocks import ModularMultiplicationBlock

mul = ModularMultiplicationBlock(7, 15)
print(mul.classical_action(np.arange(16)))

## Phase estimation, one controlled-powers kernel

With eleven ancillas, QPE's controlled-$U$ ladder is $2^{11} - 1$ applications
of the same block.  The engine recognises the run and applies
$U^{2^j}$ under ancilla $j$ from one matrix, so the phase $3/8$ of
$P(2\pi \cdot 3/8)$ reads out exactly.

In [ ]:
import qarp
from qarp.algorithms import QPE
from qarp.blocks import ComputationalBasisStateBlock
from qarp.engines import QarpEngine

u = SimpleBlock(1)
u.p(0, 2 * np.pi * 0.375)
u.build()

qpe = QPE(
    state=ComputationalBasisStateBlock([1]),
    unitary=u,
    n_ancilla=11,
    engine=QarpEngine(n_shots=qarp.EXACT),
).build()
print("phase:", qpe.run(), " probability:", qpe.result_probability)

## The knob

`structured=False` on `Block.statevector` or `QarpEngine` runs the gate
stream as-is; `QARP_STRUCTURED=0` does the same process-wide.  Engines with a
`device`, amplitude-consuming primitives and gradients always run the gate
path.